# Boltz-2 on Euler — your first run

This notebook takes you through one complete Boltz-2 run on six small example complexes, start
to finish. It takes **about 20 minutes**, most of it waiting for jobs.

By the end you will have predicted structures, a table of confidence scores, and a feel for the
steps, so you can do the same with your own molecules.

### Opening it

Run it on Euler, so everything happens on the cluster and nothing needs installing. First
copy it somewhere you can write, from an Euler shell:

```bash
cp /cluster/project/beltrao/lucla/repos/hpc-tools/boltz2/examples/01_first_run.ipynb ~/
```

The shared copy is not writable by you, so run your own: it is yours to run, edit and break.

This notebook only submits jobs and reads their results, and peaked at about 380 MB over two
full test runs. All the real work runs as separate SLURM jobs, so whatever serves the kernel
needs one core, little memory and **no GPU**; two hours is plenty.

### Something worth knowing up front

Euler's compute nodes — including the one this notebook's kernel runs on — have **no internet
access by default**. Two steps need it: fetching the example sequences, and fetching alignments.
Those load the `eth_proxy` module first. Forget it and you get a connection timeout, not a clear
error.

In [ ]:
import getpass, json, os, re, shutil, subprocess, time
from pathlib import Path

USER = getpass.getuser()

def find_tool() -> Path:
    """The hpc-tools/boltz2 directory, wherever this notebook was opened from."""
    def is_tool(p):
        return (p / "boltz2").is_file() and (p / "boltz2_run.py").is_file()
    candidates = [
        os.environ.get("HPC_TOOLS_BOLTZ2_DIR", ""),               # set explicitly
        str(Path(shutil.which("boltz2")).resolve().parent) if shutil.which("boltz2") else "",
        str(Path.cwd().parent),                                   # opened inside the checkout
        # The group's shared checkout, as named in the README's first step.
        "/cluster/project/beltrao/lucla/repos/hpc-tools/boltz2",
    ]
    for c in candidates:
        if c and is_tool(Path(c)):
            return Path(c)
    raise FileNotFoundError("cannot find hpc-tools/boltz2; set HPC_TOOLS_BOLTZ2_DIR to it")

TOOL   = find_tool()
BOLTZ2 = TOOL / "boltz2"
DRIVER = TOOL / "boltz2_run.py"

# Everything this first run writes goes under one folder on scratch, so it is easy to find and
# easy to delete. Each execution of the notebook gets its own run directory, so running it twice
# never collides with a finished run.
BASE   = Path(f"/cluster/scratch/{USER}/boltz2/first_run")
INPUTS = BASE / "inputs"
STORE  = BASE / "msa_store"          # for real work, keep your store in project storage
STAMP  = time.strftime("%Y%m%d_%H%M%S")
RUN    = BASE / f"run_{STAMP}"
DEST   = BASE / f"results_{STAMP}"
for d in (BASE, STORE, Path(f"/cluster/scratch/{USER}/boltz2/slurm")):
    d.mkdir(parents=True, exist_ok=True)

# Two helpers. Both run the command in a login shell (bash -l), because that is what sets up
# `module` and puts sbatch and squeue on PATH -- the kernel's own environment has neither.
#   run(cmd)      shows the output line by line as it arrives, and returns nothing
#   capture(cmd)  shows nothing, and returns the output, for when the next line needs it
# Both stop the notebook if the command fails.

def run(cmd: str) -> None:
    print("$", cmd, flush=True)
    p = subprocess.Popen(["bash", "-lc", cmd], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")

def capture(cmd: str) -> str:
    # A login shell first runs your own start-up files (~/.bash_profile), and anything they print
    # would land in front of the output. Everything up to this marker line is dropped.
    mark = "--- command output starts ---"
    p = subprocess.run(["bash", "-lc", f"echo '{mark}'; {cmd}"], capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout + p.stderr)
        raise RuntimeError(f"exit {p.returncode}: {cmd}")
    return p.stdout.split(mark + "\n", 1)[-1]

print("tool :", TOOL)
print("run  :", RUN)

## Step 0 — check the installation

`boltz2 info` says which image the wrapper found. `--verify` also re-checks the image's checksum
against what was recorded when it was built, which is how you know a copied image is the same
image.

In [ ]:
run(f"{BOLTZ2} info --verify")

## Step 1 — make the example inputs

`make_examples.py` fetches six small proteins from UniProt **by accession** and writes six
example inputs. No sequence is pasted into the script, so the files cannot drift from what the
accessions actually hold; rerun it any time to rebuild them.

| file | what it is |
|---|---|
| `01_monomer` | one protein, nothing else |
| `02_protein_ligand` | a protein and a small molecule, with an affinity prediction |
| `03_homodimer` | two identical copies of one protein |
| `04_protein_dna` | a protein and a double-stranded DNA fragment |
| `05_heterodimer` | two *different* proteins — the case that needs a paired alignment |
| `06_pocket_constraint` | a ligand told roughly where to sit |

One shape each — not a list of what the pipeline supports. A single input can combine them (a
protein, a DNA duplex, two ligands and a pocket constraint in one file is ordinary), and
anything valid in Boltz-2's own input format is passed through untouched.

This is the first cell that needs the internet, hence `module load eth_proxy`.

In [ ]:
run(f"module load eth_proxy && {BOLTZ2} python {TOOL}/examples/make_examples.py "
    f"--out {INPUTS} --large {BASE}/inputs_large --table {BASE}/chains.csv")

Here is one of them. This is Boltz-2's own input format: one entry per chain or molecule.

`msa: auto` is *this tool's* marker for "fill in the alignment from the store". It is not a
Boltz keyword; the `plan` step replaces it with a real file path.

In [ ]:
print((INPUTS / "02_protein_ligand.yaml").read_text()[:700])

## Step 2 — fetch the alignments

An **MSA** (multiple sequence alignment) is a set of evolutionarily related sequences lined up
against your protein. Boltz-2 reads which positions are conserved out of it, and it matters more
for the quality of the prediction than anything else you control.

This runs as a **CPU job**: it spends its whole life waiting on a web server, so holding a GPU
for it would waste one.

The store is keyed by the sequence itself, so anything fetched here is never fetched again — by
this run or any later one. For real work, put the store in project storage and keep it.

In [ ]:
msa_job = capture(f"sbatch --parsable {TOOL}/submit_msa.sh {INPUTS} {STORE}").strip().split(";")[0]
print("alignment job:", msa_job)

In [ ]:
def wait_for(jobs, poll=20, label=""):
    """Block until SLURM jobs have left the queue, printing their states as they change."""
    seen = None
    while True:
        states = capture(f"squeue -j {jobs} -h -o %T").split()
        if not states:
            break
        summary = ", ".join(f"{s} x{states.count(s)}" for s in sorted(set(states)))
        if summary != seen:
            print(f"  {time.strftime('%H:%M:%S')}  {label}{summary}", flush=True)
            seen = summary
        time.sleep(poll)
    # sacct is the only place to see how a job ENDED: squeue forgets it the moment it finishes.
    print(capture(f"sacct -j {jobs} -X -n -P --format=JobID,State,Elapsed"), end="")

wait_for(msa_job, label="alignments: ")

In [ ]:
# One file per distinct protein sequence, named by the sha1 of that sequence.
for p in sorted(STORE.glob("*.csv")):
    print(f"  {p.name}  {p.stat().st_size / 1024:6.0f} KB")
# The heterodimer also got a PAIRED alignment, cached for that combination of chains:
for p in sorted(STORE.glob("paired/*/")):
    print(f"  paired/{p.name}/")

## Step 3 — plan the run

`plan` turns the inputs into a run directory on scratch: alignments filled in, inputs grouped
into **shards** (one shard = one array task = one GPU), and the sampling options fixed. Fixing
them here rather than at submission is what makes scores from different shards comparable.

Two choices worth understanding:

- `--samples 1`. Each **diffusion sample** is an independent attempt at the structure. Real work
  here uses 5, which gives a best-of and a spread; 1 is right for "does this run at all".
- `--per-shard 2`. Inputs per array task. Small, so this example spreads over several jobs and
  you can see the sizing at work.

Everything per-prediction goes on **scratch**, never project storage: one prediction writes about
twenty files, and shared storage runs out of file-count quota long before it runs out of space.

In [ ]:
run(f"{BOLTZ2} python {DRIVER} plan --inputs {INPUTS} --run-dir {RUN} "
    f"--msa-store {STORE} --per-shard 2 --samples 1")

`plan/jobs.csv` has one row per input, always — including any that could not be planned, with
the reason. Nothing is dropped silently.

A **token** is one residue, or one heavy atom of a ligand. It is the size measure that drives
memory and time.

In [ ]:
import csv
with open(RUN / "plan" / "jobs.csv") as fh:
    for r in csv.DictReader(fh):
        print(f"  {r['job_id']:<22} {r['boltz2:n_tokens']:>5} tokens  "
              f"{r['boltz2:shard']:<12} {r['boltz2:status']}")

## Step 4 — read what will be submitted, then submit it

`plan` grouped the shards into **bands** by size, and wrote one `sbatch` line per band into
`plan/submit.sh`. Two reasons for the bands:

- Memory grows roughly with the square of the token count. A band of small complexes should not
  be charged the memory its largest neighbour needs, and that memory is taken from everyone else
  on the cluster.
- Inputs that ask for **affinity** get their own shards. Boltz-2 runs its affinity stage over
  every input in a call, and dies on any that did not ask for it — after the structures are done.

Each line says whether its memory figure has been **measured** here or only **extrapolated**.
Read the script before you run it; it is short.

In [ ]:
print((RUN / "plan" / "submit.sh").read_text())

In [ ]:
out = capture(f"bash {RUN}/plan/submit.sh")
print(out)
jobs = re.findall(r"^job (\d+)", out, flags=re.M)
print("prediction jobs:", jobs)

In [ ]:
wait_for(",".join(jobs), poll=30, label="predictions: ")

If something failed, the logs are under `/cluster/scratch/$USER/boltz2/slurm/`. The two usual
causes are running out of GPU memory (raise `--gres=gpumem:` for that band in `plan/submit.sh`
and run just that line again) and running out of time (which GPU did it land on? runtime varies
several-fold between card types).

Each shard leaves a receipt saying where it ran, on which card, and with which image:

In [ ]:
for p in sorted((RUN / "shards").glob("*.job.json")):
    j = json.loads(p.read_text())
    print(f"  {j['shard']}  {j['gpu']:<28} {j.get('elapsed_s')} s  exit {j.get('exit_code')}  "
          f"image {str(j.get('image_sha256'))[:12]}")

Look at the GPU column. Unless you asked for one model, the shards land on whatever cards are
free, and when this notebook was tested they landed on two different models. That matters in
two ways. Runtime differs between cards — which is why the walltimes in `submit.sh` carry a
margin. And Boltz-2 is not bit-reproducible across card types, so the same input can score
slightly differently on each. For a run whose scores must be strictly comparable, pin one model
by setting `EXPECT_GPU` (see `submit_boltz2.sh`).

## Step 5 — finalize

Gathers the shard tables into one scores table, copies the archives to the destination, and
writes a provenance record of what ran, with which image, on which GPUs. It refuses to run while
any shard is unfinished, so you cannot publish half a run by accident.

In [ ]:
run(f"{BOLTZ2} python {DRIVER} finalize --run-dir {RUN} --dest {DEST} --label first_run")

## Step 6 — read the results

One row per input. Higher is more confident, **except** the PDE columns (predicted distance
error, in Ångström, lower is better) and `affinity_pred_value`, which is log₁₀(IC50 in µM) — so
lower means stronger predicted binding.

Each score appears twice: the plain name is the top-ranked sample, `_mean` the mean over all
samples. With `--samples 1` they are identical, which is itself a reason to use more samples for
real work: one sample tells you nothing about how stable the answer is.

In [ ]:
COLS = ["job_id", "boltz2:status", "boltz2:n_tokens", "boltz2:confidence_score", "boltz2:ptm",
        "boltz2:iptm", "boltz2:ligand_iptm", "boltz2:complex_plddt",
        "boltz2:affinity_pred_value", "boltz2:affinity_probability_binary"]
scores_csv = DEST / "first_run_boltz2_scores.csv"
try:
    import pandas as pd
    scores = pd.read_csv(scores_csv)
    display(scores[[c for c in COLS if c in scores.columns]])
except ImportError:                       # a kernel without pandas: print the same columns
    with open(scores_csv) as fh:
        rows = list(csv.DictReader(fh))
    cols = [c for c in COLS if c in rows[0]]
    print("  ".join(c.replace("boltz2:", "") for c in cols))
    for r in rows:
        print("  ".join(r[c][:8] for c in cols))

Look at which columns are filled for which row. ipTM, the confidence in the *interfaces* between
chains, is 0 for the lone protein — it has none. `ligand_iptm` means something only where there
is a ligand, and `affinity_*` only for the one input that asked for it. A blank is the format
being honest: a zero there would read as a real, terrible score.

The pocket-constraint example usually scores noticeably lower than the plain protein-ligand one.
That is expected, and instructive: its constraint residues are numbered against a UniProt
*precursor*, signal peptide included, so they do not point at the real binding site. Always
number constraints against the sequence actually in your file.

In [ ]:
print((DEST / "first_run_boltz2_provenance.txt").read_text())

## Step 7 — get a structure out

The structures are in zips, not loose files, for the file-count reason above. A zip's index lets
you pull one file out without unpacking the rest. `model_0` is the sample Boltz ranked first.

In [ ]:
import zipfile
z = sorted(DEST.glob("first_run_boltz2_shard_*.zip"))[0]
with zipfile.ZipFile(z) as zf:
    members = zf.namelist()
    print(f"{z.name}: {len(members)} files")
    for m in members[:8]:
        print("  ", m)
    cif = next(m for m in members if m.endswith("_model_0.cif"))
    out = DEST / Path(cif).name
    out.write_bytes(zf.read(cif))
print(f"\nextracted {out}  ({out.stat().st_size / 1024:.0f} KB)")
print("Open it in ChimeraX or PyMOL.")

## Step 8 — feed the real numbers back

This is the part people skip. The resource model in `sizing.tsv` rests on very few runs. Now that
you have done one, see what it actually cost.

This runs with plain `python3`, not through the image, because `sacct` — where SLURM keeps what a
job used — exists on the cluster but not inside the image.

In [ ]:
run(f"python3 {DRIVER} size --from-run {RUN}")

If you ran a size that `sizing.tsv` marks as extrapolated, add your numbers to it with the job
ids; the file says how. That is how the estimates get better for everyone.

One caveat that matters: SLURM records **host** memory (`MaxRSS`), never GPU memory. A GPU memory
band can only ever be confirmed by jobs *not* failing at it. Raise one only after an actual CUDA
out-of-memory error.

---

## Doing this with your own molecules

Replace step 1 with your own inputs; the rest is unchanged. In a terminal, set `TOOL` first, as in the README's set-up.

1. One YAML per complex — the shapes are in the [README](../README.md#inputs), or build them from
   a table with `boltz2_run.py prep`.
2. `sbatch "$TOOL/submit_msa.sh" <your inputs> <your store>`
3. `plan` — with `--samples 5` for real work
4. read `plan/submit.sh`, then `bash` it
5. `finalize`

### Tidying up

Everything from this notebook is under one folder on scratch, which is purged periodically
anyway. The alignment store is the one thing worth keeping for real work: put yours in project
storage.

In [ ]:
print(f"everything from this notebook: {BASE}")
print(f"  this run                   : {RUN}")
print(f"  its results                : {DEST}")